# nn-parameter-wrap — ex3: frozen Parameter — requires_grad=False keeps it visible but un-trainable

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-parameter-wrap`. Running the final beacon cell reports progress against the `PyTorch: nn.Parameter` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Parameter` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-parameter-wrap`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-parameter-wrap"
DD_SUBTOPIC = "PyTorch: nn.Parameter"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## nn.Parameter — quick refresher

`nn.Parameter` is a tensor subclass that auto-registers when assigned as a Module attribute. Two independent flags govern its behavior:
1. **Visibility** — whether it shows up in `.parameters()` and `.state_dict()`. Controlled by wrapping in `nn.Parameter`.
2. **Gradient tracking** — whether autograd records ops on it. Controlled by `requires_grad` (default `True` for Parameters).

**This drill (ex3) vs prior.** ex1 contrasted Parameter vs raw tensor (visibility); ex2 contrasted Parameter vs buffer (the buffer is the alternative *registry*). ex3 holds Parameter constant and varies `requires_grad` — the canonical **frozen-weight** pattern (transfer learning, LoRA base weights). A frozen Parameter is still in the state dict but the optimizer sees no grad.

### Exercise 3 — frozen Parameter — requires_grad=False keeps it visible but un-trainable

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `nn.Parameter(tensor, requires_grad=False)` to register a frozen weight on a Module, then verify it appears in `.parameters()` and `.state_dict()` but its `.grad` stays `None` after a backward pass.
> Keywords: frozen-weight, requires-grad, transfer-learning, freeze
> ```

**KCs targeted:** `parameter-wrap-tensor`, `parameter-requires-grad-flag`

Implement `FrozenScaler` — a Module with TWO Parameters of identical shape; one trainable, one frozen.

1. Subclass `t.nn.Module`. In `__init__(self)`:
   - Call `super().__init__()` first.
   - `self.alpha = nn.Parameter(t.ones(3))` — trainable.
   - `self.beta  = nn.Parameter(t.ones(3), requires_grad=False)` — frozen.
2. `forward(self, x: Tensor) -> Tensor` returns `x * self.alpha + self.beta` (both broadcast across `x`).
3. Return an INSTANCE from `ex3_build_frozen_scaler()`.

**What this drill verifies.** After one `.backward()` call:
- `alpha.grad` is a real tensor (trainable Parameter accumulates gradient).
- `beta.grad` stays `None` (frozen — autograd never touched it).
- BOTH alpha and beta appear in `module.parameters()` and `module.state_dict()` (the wrap-with-nn.Parameter visibility is INDEPENDENT of requires_grad).

This is exactly how transfer learning works: freeze the backbone Parameters but keep them in the state dict so they save/load with the model.

In [ ]:
def ex3_build_frozen_scaler() -> 't.nn.Module':
    """Return a Module with one trainable + one frozen Parameter."""
    raise NotImplementedError()


def _test_ex3():
    import torch.nn as nn
    mod = ex3_build_frozen_scaler()
    assert isinstance(mod, t.nn.Module)

    # Both attrs must be nn.Parameter instances.
    assert isinstance(mod.alpha, nn.Parameter), f'alpha must be nn.Parameter, got {type(mod.alpha)}'
    assert isinstance(mod.beta, nn.Parameter),  f'beta must be nn.Parameter, got {type(mod.beta)}'

    # requires_grad flags split correctly.
    assert mod.alpha.requires_grad is True,  f'alpha.requires_grad must be True, got {mod.alpha.requires_grad}'
    assert mod.beta.requires_grad  is False, f'beta.requires_grad must be False, got {mod.beta.requires_grad}'

    # Both visible in .parameters() — wrapping is sufficient.
    params = list(mod.parameters())
    assert len(params) == 2, f'expected 2 Parameters listed, got {len(params)}'

    # Both visible in state_dict — saving/loading still works.
    sd_keys = set(mod.state_dict().keys())
    assert 'alpha' in sd_keys, f'alpha missing from state_dict: {sd_keys}'
    assert 'beta'  in sd_keys, f'beta missing from state_dict:  {sd_keys}'

    # Forward + backward — alpha gets grad, beta does NOT.
    x = t.tensor([1.0, 2.0, 3.0], requires_grad=False)
    y = mod(x).sum()
    y.backward()
    assert mod.alpha.grad is not None, 'alpha.grad must be populated after backward'
    assert t.allclose(mod.alpha.grad, t.tensor([1.0, 2.0, 3.0])), (
        f'alpha.grad wrong: {mod.alpha.grad}'
    )
    assert mod.beta.grad is None, f'beta.grad must remain None (frozen), got {mod.beta.grad}'

    # Optimizer filter idiom — common transfer-learning pattern.
    trainable = [p for p in mod.parameters() if p.requires_grad]
    assert len(trainable) == 1, f'expected 1 trainable Parameter, got {len(trainable)}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
import torch.nn as nn

class FrozenScaler(t.nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(t.ones(3))
        self.beta  = nn.Parameter(t.ones(3), requires_grad=False)
    def forward(self, x):
        return x * self.alpha + self.beta

def ex3_build_frozen_scaler():
    return FrozenScaler()
```

**Two orthogonal flags.** Wrapping in `nn.Parameter` controls *visibility* (parameters / state_dict). `requires_grad` controls *autograd participation*. The frozen-weight pattern combines `Parameter` + `requires_grad=False` to get visibility without gradient.

**Why not just use a buffer?** A buffer is for non-learnable state that's not even *conceptually* a parameter (running BatchNorm stats, embedding lookup tables before pretraining). A frozen Parameter is one you WILL probably unfreeze later — keeping it as a Parameter signals that intent and lets you write `p.requires_grad = True` to thaw it.

**Optimizer filter.** Always pass `filter(lambda p: p.requires_grad, model.parameters())` to your optimizer constructor when you have frozen Parameters — otherwise PyTorch silently ignores them at step time but still tracks them for state-management purposes (wasted memory).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()